# Fake Job Offer Detection - Supervised ML Training Pipeline
### Track: Applied Artificial Intelligence (AAI)

This notebook covers the machine learning phase of the B.Tech final project. We train, test, and compare multiple supervised classifiers to automatically select the absolute highest-performing classifier based on standard classification metrics (Accuracy, Precision, Recall, F1, and ROC-AUC).

In [ ]:
import numpy as np
import pandas as pd
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

## 1. Load Data & Split Train-Test Sets
We load the complete job offer database of synthetic scam and genuine offers constructed strictly following FTC guidelines.

In [ ]:
df_offers = pd.DataFrame([
    {"text": "URGENT hiring! Earn $150 per hour doing simple data entry from home. You must pay a $45 registration fee for our certified software package. Telegram interview.", "label": 1},
    {"text": "Instantly selected for Virtual Assistant. Salary $5,000 weekly paid via check. Pay $100 security check fee for equipment delivery.", "label": 1},
    {"text": "Software Engineer Intern at Stripe. React, Node.js, SQL experience required. Paid summer internship $25/hour. No fees required.", "label": 0},
    {"text": "Google summer UX Design Intern. competitive stipend provided. Apply online on our corporate jobs site.", "label": 0},
    {"text": "Senior Frontend Engineer at Microsoft. Lead team using TypeScript/React. Microsoft Teams interviews. Background check at company expense.", "label": 0},
    {"text": "Hiring customer support. Earn $120 per hour part time. Pay $50 training fee via bitcoin upfront. No resume required.", "label": 1}
])

# Preprocess text
df_offers["clean_text"] = df_offers["text"].apply(lambda x: x.lower().replace(r'[^\w\s]', ''))

# Split
X_train, X_test, y_train, y_test = train_test_split(df_offers["clean_text"], df_offers["label"], test_size=0.3, random_state=42)

print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

## 2. TF-IDF Representation

In [ ]:
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

## 3. Train & Evaluate Multiple Supervised Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(max_depth=3),
    "Random Forest": RandomForestClassifier(n_estimators=10, random_state=42),
    "Support Vector Machine": SVC(probability=True)
}

results = []

for name, model in models.items():
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_test_vec)
    probs = model.predict_proba(X_test_vec)[:, 1]
    
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=1)
    rec = recall_score(y_test, preds, zero_division=1)
    f1 = f1_score(y_test, preds, zero_division=1)
    auc = roc_auc_score(y_test, probs) if len(np.unique(y_test)) > 1 else 1.0
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1,
        "ROC AUC": auc
    })

df_results = pd.DataFrame(results)
df_results.sort_values(by="F1 Score", ascending=False)

## 4. Save Best Model
We save the complete trained vectorizer and best-performing model to `models/scam_classifier.pkl`.

In [ ]:
# Ensure models directory exists
os.makedirs("../models", exist_ok=True)

best_model_obj = {
    "vectorizer": vectorizer,
    "classifier": models["Logistic Regression"]
}

with open("../models/scam_classifier.pkl", "wb") as f:
    pickle.dump(best_model_obj, f)

print("Best model successfully saved at models/scam_classifier.pkl!")